In [17]:
import numpy
import pandas
import six
import tensorflow as tf

import CS230

# #1 MNIST online example

In [2]:
mnist = tf.keras.datasets.mnist

In [3]:
(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

11493376/11490434 [==============================] - 1s 0us/step


In [6]:
x_train.shape

(60000, 28, 28)

In [7]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [8]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [9]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
60000/60000 [==============================] - 4s 74us/sample - loss: 0.2176 - acc: 0.9362
Epoch 2/5
60000/60000 [==============================] - 4s 70us/sample - loss: 0.0954 - acc: 0.9711
Epoch 3/5
60000/60000 [==============================] - 4s 71us/sample - loss: 0.0684 - acc: 0.9782
Epoch 4/5
60000/60000 [==============================] - 4s 71us/sample - loss: 0.0530 - acc: 0.9826
Epoch 5/5
60000/60000 [==============================] - 4s 71us/sample - loss: 0.0449 - acc: 0.9853


In [10]:
model.evaluate(x_test, y_test)

10000/10000 [==============================] - 0s 17us/sample - loss: 0.0630 - acc: 0.9814


[0.06303731006655726, 0.9814]

# #2 connect my data

In [18]:
file_paths = CS230.data.get_all_file_paths()
file_path = file_paths[0]

train_percent = 0.9
dev_percent = 0.05
test_percent = 0.05

df = CS230.data.load(file_path)
df = CS230.data.stride_rows(df, stride=10)
df = CS230.data.add_derivatives(df, stride=1)
df = CS230.data.clean_discontinuities(df, stride=1)

df_train, df_dev, df_test = CS230.data.get_data_sets(df, train_percent, dev_percent, test_percent, 
                                                     CS230.data.COLUMNS_HUMAN_INPUT, CS230.data.COLUMNS_DERIV)

In [19]:
df_train.head()

,brake,throttle,handwheelAngle,deriv_axCG,deriv_ayCG,deriv_pitchAngle,deriv_pitchRate,deriv_rollAngle,deriv_rollRate,deriv_vxCG,deriv_vyCG,deriv_wheelAccelFL,deriv_wheelAccelFR,deriv_wheelAccelRL,deriv_wheelAccelRR,deriv_yawAngle,deriv_yawRate
0,0.0,3.3,3.5,1.58,-0.56,0.01,0.58,-0.03,-1.80,0.05,0.01,7.16,-1.37,2.36,-3.38,0.01,0.33
1,0.0,89.6,-3.1,2.47,-1.08,-0.04,-1.96,0.19,11.18,0.06,-0.03,-5.40,-7.36,2.65,-3.34,0.01,0.45
2,26.5,0.8,29.4,-0.81,1.61,-0.07,-4.88,0.08,2.79,-0.08,0.01,7.84,7.45,-9.03,-1.82,0.12,1.73
3,0.0,3.2,3.7,-0.04,-0.36,0.01,2.11,-0.01,-1.30,-0.01,0.00,-2.06,1.77,1.67,-1.74,0.01,0.41
4,0.0,0.7,-33.0,0.01,-0.04,0.00,0.21,0.02,0.15,0.00,0.00,0.00,-0.20,-0.30,0.15,0.02,0.39


In [26]:
df_train[CS230.data.COLUMNS_HUMAN_INPUT].values

array([[  0. ,   3.3,   3.5],
       [  0. ,  89.6,  -3.1],
       [ 26.5,   0.8,  29.4],
       ...,
       [  0. ,   0.8, -33. ],
       [  0. ,   1.3,  -9. ],
       [  0. ,   0.8, -33.1]])

In [22]:
train_dataset = (
    tf.data.Dataset.from_tensor_slices(
        (
            tf.cast(df_train[CS230.data.COLUMNS_HUMAN_INPUT].values, tf.float32),
            tf.cast(df_train[CS230.data.COLUMNS_DERIV].values, tf.float32)
        )
    )
)

In [23]:
model = tf.keras.Sequential()
# Adds a densely-connected layer with 64 units to the model:
model.add(tf.keras.layers.Dense(len(CS230.data.COLUMNS_HUMAN_INPUT), activation='relu'))
# Add another:
model.add(tf.keras.layers.Dense(len(CS230.data.COLUMNS_DERIV), activation='relu'))

In [24]:
model.compile(optimizer=tf.train.AdamOptimizer(0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [29]:
x = tf.cast(df_train[CS230.data.COLUMNS_HUMAN_INPUT].values, tf.float32)
y = tf.cast(df_train[CS230.data.COLUMNS_DERIV].values, tf.float32)

model.fit(x, y, epochs=5, steps_per_epoch=1)

Epoch 1/5
1/1 [==============================] - 0s 409ms/step - loss: nan - acc: 0.1048
Epoch 2/5
1/1 [==============================] - 0s 36ms/step - loss: nan - acc: 0.0120
Epoch 3/5
1/1 [==============================] - 0s 37ms/step - loss: nan - acc: 0.0120
Epoch 4/5
1/1 [==============================] - 0s 37ms/step - loss: nan - acc: 0.0120
Epoch 5/5
1/1 [==============================] - 0s 37ms/step - loss: nan - acc: 0.0120
